# ZED AI Temporary Colab Inference Bridge

This notebook hosts a temporary open-model inference endpoint for ZED while the permanent Lightning AI setup is pending.

Recommended Colab runtime:
- Runtime type: `GPU`
- GPU: `T4` if available

Temporary architecture:
- Netlify frontend
- Render backend
- Neon database
- Colab inference endpoint

Later, replace the Colab URL in Render with the Lightning-hosted inference URL.

## 1. Install dependencies

In [ ]:
!pip -q install transformers accelerate torch fastapi uvicorn pyngrok sentencepiece

## 2. Optional: Authenticate ngrok

If you have an ngrok auth token, paste it below. If not, skip this cell and use the anonymous tunnel.

In [ ]:
from pyngrok import ngrok

# Uncomment and replace with your token if desired:
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

## 3. Load the temporary model and create the API server

For free Colab, start with `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen2.5-3B-Instruct`.
You can change `MODEL_NAME` if the runtime has enough memory.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import threading
import uvicorn

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = "You are ZED's temporary inference backend. Provide concise, useful responses for business operations, research, and assistant workflows."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

app = FastAPI(title="ZED Temporary Inference API")

class ChatRequest(BaseModel):
    message: str
    system_prompt: str | None = None
    max_new_tokens: int = 300
    temperature: float = 0.7

def generate_reply(message: str, system_prompt: str | None, max_new_tokens: int, temperature: float):
    messages = [
        {"role": "system", "content": system_prompt or SYSTEM_PROMPT},
        {"role": "user", "content": message},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_NAME}

@app.post("/chat")
def chat(req: ChatRequest):
    reply = generate_reply(
        message=req.message,
        system_prompt=req.system_prompt,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
    )
    return {"reply": reply, "model": MODEL_NAME}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
print("ZED temporary inference server started on port 8000")

## 4. Expose the API publicly with ngrok

In [ ]:
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

## 5. Test the endpoint

In [ ]:
import requests

base_url = str(public_url)
print(requests.get(f"{base_url}/health").json())
print(requests.post(
    f"{base_url}/chat",
    json={"message": "Give me a one-sentence hello from ZED."},
).json())

## 6. Render environment notes

Because this Colab notebook exposes a generic `/chat` endpoint instead of a full Ollama API, the Render backend should temporarily use a bridge/adapter value instead of a raw Ollama URL.

If the backend is later updated to support generic remote inference, use:
- `REMOTE_INFERENCE_URL=<the ngrok public URL>`
- `REMOTE_INFERENCE_MODE=colab`

When Lightning AI is approved, replace the temporary URL with the permanent inference host.